# FoS v0.7.5 Proposal Evaluation

이 노트북은 실제 measured pilot과 controlled behavior benchmark를 결합해 제안서용 평가표를 생성한다. 현재 포함된 `full_agent` 결과는 기본적으로 heuristic backend smoke test이며, Ollama가 준비된 경우 chat backend 셀을 별도로 실행한다.


In [ ]:
from pathlib import Path
ROOT = Path.cwd()
print(ROOT)


## 1. 설치 및 테스트

In [ ]:
!pip -q install -e .
!PYTHONPATH=src pytest -q


## 2. 실제 target-pair profile 확인

In [ ]:
import json, pandas as pd
profile = json.loads(Path('outputs/real_pair_profile_v075/pair_profile.json').read_text())
pd.DataFrame(profile)


## 3. Behavior benchmark 생성

In [ ]:
!rm -rf examples/evaluation_v075/behavior_release
!PYTHONPATH=src python scripts/build_behavior_benchmark.py --output-dir examples/evaluation_v075/behavior_release --repeats-per-type 2


## 4. Proposal release 결합

In [ ]:
!rm -rf evaluation/releases/fos_eval_proposal_real_pilot_v075
!PYTHONPATH=src python scripts/assemble_proposal_evaluation_release.py \
  --measured-release evaluation/releases/egfr_her2_replay_doc3632549 \
  --behavior-release examples/evaluation_v075/behavior_release \
  --output-dir evaluation/releases/fos_eval_proposal_real_pilot_v075


## 5. Baseline 및 heuristic Full FoS 반복 실행

In [ ]:
!rm -rf outputs/proposal_eval_real_v075/runs
!PYTHONPATH=src python scripts/run_evaluation_suite_v2.py \
  --episodes evaluation/releases/fos_eval_proposal_real_pilot_v075/public/proposal_episodes.jsonl \
  --action-spaces evaluation/releases/fos_eval_proposal_real_pilot_v075/manifests/proposal_action_spaces.jsonl \
  --policies seed_only,greedy,tool_only,full_agent \
  --full-agent-backend heuristic \
  --output-dir outputs/proposal_eval_real_v075/runs


## 6. 지표 및 제안서 표 생성

In [ ]:
!rm -rf outputs/proposal_eval_real_v075/report
!PYTHONPATH=src python scripts/generate_evaluation_report.py \
  --episodes evaluation/releases/fos_eval_proposal_real_pilot_v075/public/proposal_episodes.jsonl \
  --action-spaces evaluation/releases/fos_eval_proposal_real_pilot_v075/manifests/proposal_action_spaces.jsonl \
  --oracles evaluation/releases/fos_eval_proposal_real_pilot_v075/private_oracle/proposal_oracle.jsonl \
  --runs outputs/proposal_eval_real_v075/runs \
  --output-dir outputs/proposal_eval_real_v075/report \
  --pilot-label 'FoS v0.7.5 proposal pilot' \
  --dataset-status '1 real measured episode + 12 controlled behavior fixtures; heuristic full-agent smoke test'


In [ ]:
from IPython.display import display, Markdown
display(Markdown(Path('outputs/proposal_eval_real_v075/report/MEASURED_PILOT_RESULT_TABLE.md').read_text()))
display(Markdown(Path('outputs/proposal_eval_real_v075/report/BEHAVIOR_PILOT_RESULT_TABLE.md').read_text()))


## 7. Development calibration smoke test

In [ ]:
!rm -rf outputs/calibration_real_v075
!PYTHONPATH=src python scripts/calibrate_agent_policy.py \
  --episodes evaluation/releases/fos_eval_proposal_real_pilot_v075/public/proposal_episodes.jsonl \
  --action-spaces evaluation/releases/fos_eval_proposal_real_pilot_v075/manifests/proposal_action_spaces.jsonl \
  --oracle evaluation/releases/fos_eval_proposal_real_pilot_v075/private_oracle/proposal_oracle.jsonl \
  --output-dir outputs/calibration_real_v075 \
  --policy tool_only \
  --min-delta-s-grid 0.5,1.0 \
  --min-delta-on-grid=-0.75,-0.5 \
  --support-grid 1,2 \
  --top-k-grid 3,5 \
  --max-iterations-grid 3 \
  --provisional-grid true,false


## 8. Qwen/Ollama Full agent 실행 (Ollama가 켜진 환경에서만)

In [ ]:
# 아래 셀은 Ollama endpoint가 준비된 경우 주석을 해제한다.
# !PYTHONPATH=src python scripts/run_evaluation_suite_v2.py \
#   --episodes evaluation/releases/fos_eval_proposal_real_pilot_v075/public/proposal_episodes.jsonl \
#   --action-spaces evaluation/releases/fos_eval_proposal_real_pilot_v075/manifests/proposal_action_spaces.jsonl \
#   --policies full_agent --full-agent-backend chat \
#   --model qwen3:8b --base-url http://127.0.0.1:11434/v1 \
#   --output-dir outputs/proposal_eval_qwen_v075/runs


## 해석 주의

- measured 표는 실제 ChEMBL held-out replay이지만 episode가 1건이다.
- behavior 표는 agent-control fixture이며 생물학적 활성 결과가 아니다.
- heuristic `full_agent`와 `tool_only`의 동일 성능은 LLM 우월성을 뜻하지 않는다.
- Qwen/API 및 Stage C 독립 predictor 결과는 별도로 추가해야 한다.
